In [ ]:
import os
import sys
import joblib
import logging
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)

logging.info("Logging is working ✅")


SAVE_DIR = "/content/drive/MyDrive/PhytoLB/model_enhanced_v2"
os.makedirs(SAVE_DIR, exist_ok=True)

logging.info(f"Saving outputs to: {SAVE_DIR}")


logging.info("Loading GBIF dataset...")

gbif = load_gbif_txt(
    "/content/drive/MyDrive/PhytoLB/dwca-tree_lebanon-v1.0/occurrence.txt"
)

logging.info(f"GBIF loaded: {len(gbif)} records")
logging.info(f"Unique species: {gbif['species'].nunique()}")

gbif.to_csv(f"{SAVE_DIR}/clean_gbif.csv", index=False)
logging.info("Clean GBIF dataset saved ✅")


logging.info("Initializing Climate Extractor...")

extractor = ClimateExtractor(
    "/content/drive/MyDrive/PhytoLB/wc2.1_10m_bio"
)

logging.info("Building dataset with climate features...")
dataset = build_dataset(gbif, extractor)

logging.info(f"Dataset built: {dataset.shape}")
logging.info(f"Class distribution:\n{dataset['label'].value_counts()}")

missing = dataset.isna().sum().sum()
logging.info(f"Total missing values in dataset: {missing}")

# Save extracted dataset
dataset.to_csv(f"{SAVE_DIR}/final_phyto_dataset.csv", index=False)
dataset.to_parquet(f"{SAVE_DIR}/final_phyto_dataset.parquet", index=False)

logging.info("Final extracted dataset saved ✅")


logging.info("Splitting dataset...")

X = dataset.drop(columns=["label"])
y = dataset["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logging.info(f"Train size: {X_train.shape}")
logging.info(f"Test size: {X_test.shape}")

# Save raw split before transformations
X_train.to_csv(f"{SAVE_DIR}/X_train_raw.csv", index=False)
X_test.to_csv(f"{SAVE_DIR}/X_test_raw.csv", index=False)
y_train.to_csv(f"{SAVE_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{SAVE_DIR}/y_test.csv", index=False)

logging.info("Raw train/test splits saved ✅")


logging.info("Encoding species...")

le = LabelEncoder()

X_train = X_train.copy()
X_test = X_test.copy()

X_train["species_id"] = le.fit_transform(X_train["species"])
X_test["species_id"] = le.transform(X_test["species"])

X_train = X_train.drop(columns=["species"])
X_test = X_test.drop(columns=["species"])

logging.info(f"Encoded {len(le.classes_)} species")

feature_names = X_train.columns.tolist()


logging.info("Imputing missing values...")

imputer = SimpleImputer(strategy="mean")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_names,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_names,
    index=X_test.index
)

logging.info(f"NaNs after imputation train: {X_train_imputed.isna().sum().sum()}")
logging.info(f"NaNs after imputation test: {X_test_imputed.isna().sum().sum()}")

# Save processed splits
X_train_imputed.to_csv(f"{SAVE_DIR}/X_train_processed.csv", index=False)
X_test_imputed.to_csv(f"{SAVE_DIR}/X_test_processed.csv", index=False)

logging.info("Processed train/test datasets saved ✅")


logging.info("Training model...")

model = get_model()
model.fit(X_train_imputed, y_train)

logging.info("Model training completed ✅")

# -----------------------
# 7. Save Model Artifacts
# -----------------------
logging.info("Saving model artifacts...")

joblib.dump(model, f"{SAVE_DIR}/phyto_lightgbm_model.pkl")
joblib.dump(le, f"{SAVE_DIR}/species_encoder.pkl")
joblib.dump(imputer, f"{SAVE_DIR}/imputer.pkl")
joblib.dump(feature_names, f"{SAVE_DIR}/feature_names.pkl")

logging.info("Model artifacts saved ✅")

# -----------------------
# 8. Evaluate
# -----------------------
logging.info("Evaluating model...")

y_pred = model.predict(X_test_imputed)
y_prob = model.predict_proba(X_test_imputed)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

logging.info(f"Accuracy: {acc:.4f}")
logging.info(f"AUC: {auc:.4f}")

report = classification_report(y_test, y_pred)
print("\nClassification Report:\n")
print(report)

# Save evaluation results
with open(f"{SAVE_DIR}/evaluation_report.txt", "w") as f:
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"AUC: {auc:.4f}\n\n")
    f.write(report)

logging.info("Evaluation report saved ✅")
logging.info("Pipeline completed successfully 🚀")

2026-05-10 05:06:54,011 - INFO - Logging is working ✅
2026-05-10 05:06:54,029 - INFO - Saving outputs to: /content/drive/MyDrive/PhytoLB/model_enhanced_v2
2026-05-10 05:06:54,031 - INFO - Loading GBIF dataset...
2026-05-10 05:06:54,255 - INFO - GBIF loaded: 13748 records
2026-05-10 05:06:54,258 - INFO - Unique species: 27
2026-05-10 05:06:54,367 - INFO - Clean GBIF dataset saved ✅
2026-05-10 05:06:54,377 - INFO - Initializing Climate Extractor...
2026-05-10 05:06:54,674 - INFO - Building dataset with climate features...
2026-05-10 05:06:57,152 - INFO - Processed 0/13748 points
2026-05-10 05:07:34,941 - INFO - Processed 1000/13748 points
2026-05-10 05:08:04,497 - INFO - Processed 2000/13748 points
2026-05-10 05:08:33,643 - INFO - Processed 3000/13748 points
2026-05-10 05:09:04,267 - INFO - Processed 4000/13748 points
2026-05-10 05:09:33,727 - INFO - Processed 5000/13748 points
2026-05-10 05:10:04,348 - INFO - Processed 6000/13748 points
2026-05-10 05:10:34,251 - INFO - Processed 7000/13

In [ ]:
def predict_species_suitability(city_name, lat, lon, species_name):
    # Extract climate values
    climate_values = extractor.extract_point(lat, lon)

    sample_df = pd.DataFrame(
        [climate_values],
        columns=[f"BIO{i}" for i in range(1, 20)]
    )

    # Encode species
    sample_df["species_id"] = le.transform([species_name])[0]

    # Ensure correct column order
    sample_df = sample_df[feature_names]

    # Impute
    sample_df = pd.DataFrame(
        imputer.transform(sample_df),
        columns=feature_names
    )

    # Predict
    prob = model.predict_proba(sample_df)[0][1]
    label = int(prob >= 0.5)

    print(f"City: {city_name}")
    print(f"Species: {species_name}")
    print(f"Suitability Probability: {prob:.4f}")
    print(f"Prediction: {'Suitable' if label == 1 else 'Not suitable'}")

    return prob, label

In [ ]:
predict_species_suitability(
    city_name="Bcharre",
    lat=34.2500,
    lon=36.0167,
    species_name="Cedrus libani"
)

City: Bcharre
Species: Cedrus libani
Suitability Probability: 0.4046
Prediction: Not suitable


(np.float64(0.40457052801593424), 0)

In [ ]:
predict_species_suitability(
    city_name="Cedars of God",
    lat=34.2436,
    lon=36.0486,
    species_name="Cedrus libani"
)

City: Cedars of God
Species: Cedrus libani
Suitability Probability: 0.4046
Prediction: Not suitable


(np.float64(0.40457052801593424), 0)

In [ ]:
[x for x in le.classes_ if "Cedrus" in x]

['Cedrus libani']